This code is for plotting raw GNSS data from OVSICORI, and create a rates file from raw data

In [1]:
## import modules
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from io import StringIO

In [2]:
### define parameters
analysisCenter = 'ovsi'  # or 'cwu' or 'unr'

## directories 
base_dir = os.getcwd()
datadir = os.path.join(base_dir, 'POS-FILES_1days-sol', ) #'not_used_stations'
data_outdir = os.path.join(base_dir,'results/OVSI_raw_plots')
os.makedirs(data_outdir, exist_ok=True)

# Define stations
Stats=('ABEJ','AROL', 'BIJA', 'BON2', 'BRBR', 'CABA', 'CAPO', 'CDME', 'CDTO', 'CHLS', 'CHOM',  'COBB', 'COVE', 'CRU2', 'CTCR',
        'EART', 'ELVI', 'GRZA', 'HATI', 'HORI', 'IND1', 'JACO', 'LAFE', 'LBRA', 'LIMN', 'LMNL', 'LOL2', 'MUEL', 'OCHA', 'OROT', 'OVSI',
        'PAND', 'PEZE', 'PLAN', 'PN01', 'PNE2', 'POTG', 'PUJE', 'PUMO', 'PUNT', 'QPSB', 'QSEC', 'RIDC', 'RIFO', 'SAJU', 'SRBA', 
        'VERA', 'VISP', 'VRAI') # for rates
#Stats=('CIQU', 'TENO')
### stations not used  'CIQU', 'TENO',

ratesfile='CR_rates_.txt' # this file will be created

In [3]:
## get times series into Pandas dataframe
## read all the .pos into a dictionary of dataframes
def GNSSTimeSeries2Pandas(stat,analysisCenter='ovsi'):
    filename = os.path.join(datadir, f"{stat}.ovs.final_igb14.pos")
    with open(filename, 'r') as f:
        lines = f.readlines()

        data_lines = [line for line in lines if re.match(r'^\s*\d{8}\s+\d{6}', line)]

        df = pd.read_csv(StringIO("".join(data_lines)), sep=r'\s+', header=None,
                        names=[
                        'YYYYMMDD','HHMMSS','JJJJJ.JJJJ','X','Y','Z',
                        'Sx','Sy','Sz','Rxy','Rxz','Ryz',
                        'NLat','Elong','Height',
                        'dN','dE','dU','Sn','Se','Su',
                        'Rne','Rnu','Reu','Soln'
                    ])
        df['Date'] = pd.to_datetime(df['YYYYMMDD'].astype(str), format='%Y%m%d', errors='coerce')
        df['NumDate'] = df['Date'].apply(lambda x: x.year + (x.dayofyear - 1)/365.25)
        print(df[['Date', 'NumDate']].head())

        #rename for better understanding
        df['Npos'] = df['dN']
        df['Epos'] = df['dE']
        df['Upos'] = df['dU']
        df['Nerr'] = df['Sn']
        df['Eerr'] = df['Se']
        df['Uerr'] = df['Su']

        # add station name as a column
        df['STAT'] = stat

    return df




In [4]:
def parseOVSITimeSeries(df):
    if df is None or df.empty:
        print("ERROR: dataframe empty or not valid.")
        return None, None

    Comps = ('N', 'E', 'U')
    for comp in Comps:
        ycol = comp + 'pos'
        df[ycol] = df[ycol] - df[ycol].mean()

    dfFinal = df[df['Soln'] == 'final'].copy()
    dfRapid = pd.DataFrame()
    return dfFinal, dfRapid

In [5]:
def wcorr(df):
    """
    This function computes the weighted correlation coefficients between the GNSS residual components N, E, U. 
    It first builds weighted observations using the provided displacements (dN, dE, dU) and their weights (wN, wE, wU), then forms a weighted covariance matrix.
    From this, it derives the correlation coefficients for N-E, N-U, and E-U by normalizing the covariances with the corresponding variances.
    """
    Q=np.array(df[['dN','dE','dU']])
    W=np.array(df[['wN','wE','wU']])
    QW=Q*W
    C=QW.T.dot(QW)/W.T.dot(W)  # covariance
    CorrNE=C[0][1]/(C[0][0]*C[1][1])**0.5
    CorrNU=C[0][2]/(C[0][0]*C[2][2])**0.5
    CorrEU=C[1][2]/(C[1][1]*C[2][2])**0.5
    return CorrNE,CorrNU,CorrEU

In [6]:
### get information of stations from the .pos files
def getOVSIlocs(stat,analysisCenter='ovsi',refFrame='igs14'): 
    file = os.path.join(datadir, f"{stat}.ovs.final_igb14.pos")
    with open(file, 'r') as fh:
        for line in fh:
            if line.startswith("NEU Reference position"):
                words = line.split()
                lat = float(words[4])
                lon = float(words[5]) - 360.0
                height = float(words[6])
                return lat, lon, height
    raise FileNotFoundError(f"Reference position not found in {file}")

    # stations = []   
    # lats = []
    # lons = []
    # heights = []
    
    # for stat in Stats:
    #     file = os.path.join(datadir, stat + '.ovs.final_igb14.pos')
    #     with open(file, 'r') as csv_file:
    #         for line in csv_file:
    #             if line.startswith("NEU Reference position"):
    #                 words = line.split()
    #                 lat, lon, height = float(words[4]), float(words[5]) - 360, float(words[6])
    #                 lats.append(lat)
    #                 lons.append(lon)
    #                 heights.append(height)
    #                 stations.append(stat)
    #                 #print(f"Station {stat} → lat: {lat}, lon: {lon}, height: {height}")
    #                 break  # Exit after finding the reference position

    # # Create a DataFrame for station reference positions
    # station_coords = pd.DataFrame({
    #     'Station': stations,
    #     'Latitude': lats,
    #     'Longitude': lons,
    #     'Height': heights
    # })
    # return lats, lons, heights

In [10]:
### calculate raw rates from time series
## get the slopes of the time series and save to a file

def raw_rates(Stats, analysisCenter = "ovsi"):
    yscale=1000 # to get to mm displacements

    ratesFile=open(os.path.join(base_dir,'results/rates', ratesfile), 'w')  # new velocity file
    ratesFile.write('# Adjust velocities relative to the CARIB08 plate model\n')
    ratesFile.write('# Velocities are in mm/yr and referenced to ITRF14\n')
    ratesFile.write('# STAT        Lat      Long   Height      Nvel     Evel   Uvel     Nerr     Eerr     Uerr    CorrNE CorrNU CorrEU Sdate       Edate     InstallYear\n') 
    ratesFile.write('#           °        °      m          mm/yr    mm/yr    mm/yr    mm/yr    mm/yr     mm/yr                      YEAR-MO-DY   YEAR-MO-DY  YEAR-MO-DY\n') 
    ratesFile.write('#--------------------------------------------------------------------------------------------------------------------------------------------------------------\n') 
    fitDict = {}  # dictionary of fit values for individual stations and components.  this info is returned for plotting later
    for stat in Stats:
        #creating pandas readable csv from the url
        if analysisCenter == 'ovsi':
            lat, lon, height = getOVSIlocs(stat)  
        else:
            print("Could not get coordinates from "+analysisCenter+" for "+stat+", setting to defaults.")

        df=GNSSTimeSeries2Pandas(stat,analysisCenter=analysisCenter) 
        installYear = df['Date'].iloc[0].year
        
        dfFinal,dfRapid=parseOVSITimeSeries(df)
        #dfFinal = dfFinal[dfFinal['Date'] >= pd.Timestamp("2013-01-01")].copy() # filter data from 2013 onwards
        dfFinal.reset_index(drop=True, inplace=True)
        
        # save processed file per station 
        ts_results_dir = os.path.join(base_dir, 'results/processed_ts')
        os.makedirs(ts_results_dir, exist_ok=True)
        out_file = os.path.join(ts_results_dir, f"{stat}_processed.csv")
        dfFinal.to_csv(out_file, index=False)
    
        xFND=dfFinal['NumDate'] # convert to decimal years 
        sp=0
        slopes=np.zeros(3) # store slopes and errors for N,E,U
        errs=np.zeros(3) # store slopes and errors for N,E,U
        dfFNEU=pd.DataFrame() # detrended data for covariance determination
        Comps=('N','E', 'U') 
        for comp in Comps:
            ycolComp=str(comp+'pos') #Npos, Epos, or Upos
            wcolComp=str(comp+'err') #Nerr, Eerr, or Uerr
            sdate=dfFinal.Date[0].strftime("%Y-%m-%d") # start date
            edate=dfFinal.Date[len(dfFinal)-1].strftime("%Y-%m-%d") # end date
            yF=dfFinal[ycolComp]*yscale # convert to mm
            wghtF=1/dfFinal[wcolComp]/yscale #weights for fit for np.polyfit
            # performing linear fits to data
            fit,var=np.polyfit(xFND,yF,1,w=wghtF, full=False, cov=True) # linear fit to data
            fitDict[stat+'-'+comp+'-fit'] = fit
            fitDict[stat+'-'+comp+'-var'] = var
            fity=np.polyval(fit,xFND)
            errs[sp]=np.sqrt(var[0][0])
            slopes[sp]=fit[0]
            dfFNEU['d'+comp]=(yF-fity) # detrended sln for covariance determination
            dfFNEU['w'+comp]=(wghtF)   #  weights 
            sp+=1
        CorrNE,CorrNU,CorrEU=wcorr(dfFNEU) # weighted covariance
        # pAZ,pRate = Euler.velocity(lat,lon) 
        # pNvel = pRate * np.cos(np.radians(pAZ)) 
        # pEvel = pRate * np.sin(np.radians(pAZ))
        # print("Euler Pole: %s (%5.1f,%6.1f): N= %5.1f, E= %5.1f (Mag = %5.1f, %6.1f°)"
        #     %(stat,lat,lon,pNvel,pEvel,pRate,pAZ))
        ratesFile.write(f"{stat:4s} "
                        f"{lat:8.4f} {lon:9.4f} {height:9.3f}  "
                        f"{slopes[0]:9.3f} {slopes[1]:9.3f} {slopes[2]:9.3f}  "
                        f"{errs[0]:9.3f} {errs[1]:9.3f} {errs[2]:9.3f}  "
                        f"{CorrNE:7.4f} {CorrNU:7.4f} {CorrEU:7.4f}    "
                        f"{sdate}  {edate}  {installYear}\n")
    ratesFile.close()
    return fitDict

In [8]:
def timeSeriesPlots(Stats, fitDict, analysisCenter, yscale=1000):
    """
    Create displacement time series from the data pulled from GNSS analysis centers.
    This version uses only FINAL solutions.
    """
    # Plot styles
    csize = 2; elw = 0.8; ecol = 'k'      # errorbar style
    mec = 'k'; mew = 0.8; mfmt = 'o'; msz = 4  # markers
    falpha = 1; fcol = 'blue'             # final solutions
    galpha = 0.5; gvwidth = 0.5; ghwidth = 1; gcol = 'gray'  # grid

    for stat in Stats:
        # Load time series (CWU/UNR)
        df = GNSSTimeSeries2Pandas(stat, analysisCenter=analysisCenter)
        dfFinal, _ = parseOVSITimeSeries(df)  # ignore dfRapid completely

        # Skip if empty
        if dfFinal.empty:
            print(f"[WARN] {stat}: no FINAL data available — skipping.")
            continue

        # Prepare plotting variables
        f, ax = plt.subplots(3, 1, figsize=(14, 8), sharey=False, sharex=True)
        f.tight_layout(h_pad=0)

        xF = dfFinal['Date']
        xFND = dfFinal['NumDate']

        slopes = np.zeros(3)
        errs = np.zeros(3)
        dfFNEU = pd.DataFrame()

        Comps = ('U', 'N', 'E')
        for sp, comp in enumerate(Comps):
            ycol = comp + 'pos'
            wcol = comp + 'err'

            yF = dfFinal[ycol] * yscale
            yeF = dfFinal[wcol] * yscale
            wghtF = 1 / dfFinal[wcol] / yscale

            # Fit line using precomputed slopes from fitDict
            fit = fitDict[stat + '-' + comp + '-fit']
            var = fitDict[stat + '-' + comp + '-var']

            errs[sp] = np.sqrt(var[0][0])
            fity = np.polyval(fit, xFND)
            slopes[sp] = fit[0]

            # Plot FINAL data (blue)
            ax[sp].errorbar(
                x=xF, y=yF, yerr=yeF,
                fmt=mfmt, ms=msz, capsize=csize,
                label='Final', mfc=fcol, mec=mec, mew=mew,
                ecolor=ecol, elinewidth=elw, alpha=falpha
            )
            ax[sp].plot(xF, fity, color='black', linewidth=1.2)

            # Grid and labels
            ax[sp].grid(axis='x', linestyle='-', color=gcol, linewidth=gvwidth, alpha=galpha)
            ax[sp].axhline(0, linestyle='-', color=gcol, linewidth=ghwidth, alpha=galpha)

            if sp == 0:
                ax[sp].legend(loc='upper left', fancybox=True, shadow=True)
                ax[sp].set_ylabel('Up [mm]')
            elif sp == 1:
                ax[sp].set_ylabel('North [mm]')
            else:
                ax[sp].set_ylabel('East [mm]')
                ax[sp].set_xlabel('Date')
                [xmin, xmax, ymin, ymax] = plt.axis()
                ax[sp].text(
                    xmin + (xmax - xmin) * 0.005,
                    ymin + (ymax - ymin) * 0.01,
                    f"Daily positions processed @ {analysisCenter.upper()}",
                    horizontalalignment='left', verticalalignment='bottom'
                )
                ax[0].set_title(
                    f"Costa Rica Sliver Project: {stat}  "
                    f"Rates: N={slopes[0]:.1f}±{errs[0]:.1f}, "
                    f"E={slopes[1]:.1f}±{errs[1]:.1f}, "
                    f"V={slopes[2]:.1f}±{errs[2]:.1f} [mm/yr]"
                )

        # Save plot
        f.savefig(
            os.path.join('/Users/lhylu/Library/CloudStorage/OneDrive-Personal/CODING/GNSS.processing/results/OVSI_andy_plots', f"{stat}_TS.png"),
            dpi=150, facecolor='white', bbox_inches='tight', pad_inches=0.5
        )
        plt.close('all')


In [11]:
fitDict = raw_rates(Stats,analysisCenter = "ovsi")
timeSeriesPlots(Stats, fitDict, analysisCenter = "ovsi")

        Date      NumDate
0 2018-05-31  2018.410678
1 2018-06-01  2018.413415
2 2018-06-02  2018.416153
3 2018-06-03  2018.418891
4 2018-06-04  2018.421629
        Date      NumDate
0 2000-02-16  2000.125941
1 2000-03-03  2000.169747
2 2000-03-04  2000.172485
3 2000-03-05  2000.175222
4 2000-03-06  2000.177960
        Date      NumDate
0 2009-08-21  2009.635181
1 2009-08-22  2009.637919
2 2009-08-23  2009.640657
3 2009-08-24  2009.643395
4 2009-08-25  2009.646133
        Date      NumDate
0 2004-10-14  2004.785763
1 2004-10-15  2004.788501
2 2004-10-16  2004.791239
3 2004-10-17  2004.793977
4 2004-10-18  2004.796715
        Date      NumDate
0 2021-01-01  2021.000000
1 2021-01-02  2021.002738
2 2021-01-03  2021.005476
3 2021-01-04  2021.008214
4 2021-01-05  2021.010951
        Date      NumDate
0 2009-07-04  2009.503765
1 2009-07-05  2009.506502
2 2009-07-06  2009.509240
3 2009-07-07  2009.511978
4 2009-07-08  2009.514716
        Date      NumDate
0 2021-09-19  2021.714579
1 2021-10-10